# 06 Evaluating Your Model

## Overview

This notebook shows how to judge whether your fine tune actually helped.
You will compare manual review with a few lightweight automated metrics, then see how to merge and save an adapter for deployment.

Estimated time: 25 to 35 minutes.

## Prerequisites

Complete notebooks 04 and 05 first.
You should have at least one saved adapter directory to inspect.

In [ ]:
%pip install -q transformers==4.40.0 peft==0.10.0 evaluate==0.4.1 rouge_score==0.1.2 torch==2.2.2

## Why Loss Is Not Enough

Low loss can still hide bad outputs.
A model can memorize the training style, produce bland answers, or drift off topic while still showing a better loss number.

That is why you should look at real prompts and real outputs.

In [ ]:
import math
import torch
import evaluate
from pathlib import Path
import sys
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

sys.path.append(str(Path("..").resolve()))

from utils.helpers import LORA_ADAPTER_DIR, MERGED_MODEL_DIR

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = LORA_ADAPTER_DIR
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
test_prompts = [
    "Explain what a parameter is in plain English.",
    "Why is a validation split useful?",
    "What does LoRA change compared with full fine tuning?",
]

grading_checklist = [
    "Did the answer follow the instruction?",
    "Did it stay on topic?",
    "Was it clear for a beginner?",
    "Did it avoid obvious hallucinations?",
]

print("Manual evaluation checklist:")
for item in grading_checklist:
    print(f"  * {item}")

In [ ]:
rouge = evaluate.load("rouge")
predictions = [
    "A parameter is a learned number inside a model.",
    "A validation split helps test generalization on held out examples.",
]
references = [
    "A parameter is a number the model learns during training.",
    "A validation split checks performance on examples not used for training.",
]
rouge_result = rouge.compute(predictions=predictions, references=references)
rouge_result

In [ ]:
loss_value = 2.3
perplexity = math.exp(loss_value)
print(f"Example perplexity from loss {loss_value}: {perplexity:.2f}")

In [ ]:
def generate_text(current_model, prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = current_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text[len(prompt):].strip()

base_model = AutoModelForCausalLM.from_pretrained(model_name)

if adapter_path.exists() and any(adapter_path.iterdir()):
    adapted_model = PeftModel.from_pretrained(base_model, adapter_path)
    merged_model = adapted_model.merge_and_unload()
    merged_path = MERGED_MODEL_DIR
    merged_model.save_pretrained(merged_path)
    tokenizer.save_pretrained(merged_path)
    print(f"Merged model saved to {merged_path}")

    prompt = "### Instruction:\nExplain what overfitting is for a beginner.\n\n### Input:\n\n### Response:\n"
    print(generate_text(merged_model, prompt))
else:
    print("No adapter found yet. Run notebook 04 first, then come back to merge and save the result.")

## Optional Push To Hugging Face Hub

Once you have a merged model or adapter you are happy with, you can push it with the standard Transformers save and upload flow.

```python
# tokenizer.push_to_hub("your-user/your-model-name")
# merged_model.push_to_hub("your-user/your-model-name")
```

## Key Takeaways

* Loss is useful, but it is not the whole story.
* Manual prompt review catches issues metrics miss.
* ROUGE and perplexity can support your judgment for the right task.
* `merge_and_unload()` turns a base model plus adapter into one deployable model.

## Next Steps

You have reached the end of the main tutorial.
Revisit notebook 04 with your own dataset and compare the new outputs against the baseline.